## 1. Imports & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [5]:
df= pd.read_csv('dataset/heart_failure_clinical_records_dataset.csv')
y_target = 'DEATH_EVENT'
print(df.head(5))

    age  anaemia  creatinine_phosphokinase  diabetes  ejection_fraction  \
0  75.0        0                       582         0                 20   
1  55.0        0                      7861         0                 38   
2  65.0        0                       146         0                 20   
3  50.0        1                       111         0                 20   
4  65.0        1                       160         1                 20   

   high_blood_pressure  platelets  serum_creatinine  serum_sodium  sex  \
0                    1  265000.00               1.9           130    1   
1                    0  263358.03               1.1           136    1   
2                    0  162000.00               1.3           129    1   
3                    0  210000.00               1.9           137    1   
4                    0  327000.00               2.7           116    0   

   smoking  time  DEATH_EVENT  
0        0     4            1  
1        0     6            1  
2       

## 2. Handle Missing / Invalid Values

In [4]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Duplicates: 0
Missing Feature:
[]


## 3. Feature Engineering

In [13]:
df['comorbidity_count'] = df['anaemia'] +  df['diabetes'] + df['high_blood_pressure']
df['unhealthy_lifestyle'] = ((df['smoking'] == 1) & ((df['high_blood_pressure'] == 1) | (df['diabetes'] == 1))).astype(int)

df['is_elderly'] = (df['age'] >= 65).astype(int)
df['low_ejection_fraction'] = (df['ejection_fraction'] < 40).astype(int) # 1 jika di bawah 40% (Abnormal), 0 jika normal
df['high_serum_creatinine'] = (df['serum_creatinine'] > 1.3).astype(int) # Menyederhanakan indikator kreatinin tinggi secara umum

df['creatinine_sodium_ratio'] = df['serum_creatinine'] / df['serum_sodium'] # Rasio antara kreatinin dan sodium sering kali mencerminkan beban kerja ginjal dan keseimbangan cairan tubuh yang buruk pada pasien gagal jantung.
df['age_creatinine_product'] = df['age'] * df['serum_creatinine'] # Semakin tua dan semakin tinggi kadar kreatinin, risiko umumnya meningkat secara non-linear.

## 4. Analisis & Handling Outliers

In [14]:
feature_numerik = df.select_dtypes(include=np.number).columns
Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 117
Jumlah Outlier Sesudah Dihapus: 0


## 5.Save Cleaned Dataset

In [15]:
file_path = 'dataset/heart_failure_clinical_records_CLEANING.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')
df.head()

File belum ada. Berhasil menyimpan dataset baru!


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT,comorbidity_count,unhealthy_lifestyle,is_elderly,low_ejection_fraction,high_serum_creatinine,creatinine_sodium_ratio,age_creatinine_product
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1,1,0,1,1,1,0.014615,142.5
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1,0,0,1,1,0,0.010078,84.5
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1,1,0,0,1,1,0.013869,95.0
6,75.0,1,246,0,15,0,127000.00,1.2,137,1,0,10,1,1,0,1,1,0,0.008759,90.0
8,65.0,0,157,0,65,0,263358.03,1.5,138,0,0,10,1,0,0,1,0,1,0.010870,97.5
